# Cost Model

This notebook estimates the main platform cost drivers:

- Kafka / MSK throughput and retention
- EMR Flink runtime
- S3 Iceberg storage
- Redshift Serverless RPU usage

The numbers are directional only. Replace the assumptions with real cloud pricing for your AWS region.

In [ ]:
from dataclasses import dataclass
import pandas as pd

HOURS_PER_MONTH = 24 * 30


@dataclass
class CostAssumptions:
    emr_master_hourly: float = 0.192   # example m5.xlarge on-demand placeholder
    emr_core_hourly: float = 0.384     # example m5.2xlarge on-demand placeholder
    emr_core_count: int = 2
    emr_hours_per_day: float = 24
    s3_gb_month: float = 0.023
    iceberg_storage_gb: float = 250
    redshift_rpu_hour_cost: float = 0.36   # Serverless: ~$0.36 per RPU-hour (8 RPU base)
    redshift_rpu_hours_per_day: float = 8
    msk_serverless_monthly_placeholder: float = 300

assumptions = CostAssumptions()
assumptions

In [ ]:
def estimate_monthly_cost(a: CostAssumptions) -> pd.DataFrame:
    emr_hourly = a.emr_master_hourly + (a.emr_core_hourly * a.emr_core_count)
    emr_monthly = emr_hourly * a.emr_hours_per_day * 30
    s3_monthly = a.s3_gb_month * a.iceberg_storage_gb
    redshift_monthly = a.redshift_rpu_hour_cost * a.redshift_rpu_hours_per_day * 30

    rows = [
        ("EMR Flink", emr_monthly, "master + core nodes"),
        ("S3 Iceberg", s3_monthly, "bronze/silver/gold storage only"),
        ("Redshift Serverless", redshift_monthly, "RPU-hours while active"),
        ("MSK Serverless", a.msk_serverless_monthly_placeholder, "placeholder until measured"),
    ]
    df = pd.DataFrame(rows, columns=["component", "monthly_usd", "note"])
    df.loc[len(df)] = ["Total", df["monthly_usd"].sum(), "estimated monthly run-rate"]
    return df

estimate_monthly_cost(assumptions)

In [ ]:
# Scenario: part-time EMR for development / demos.
dev_assumptions = CostAssumptions(
    emr_hours_per_day=4,
    iceberg_storage_gb=25,
    redshift_rpu_hours_per_day=1,
    msk_serverless_monthly_placeholder=75,
)

estimate_monthly_cost(dev_assumptions)

## Cost Controls

Recommended controls from the architecture:

- Keep the Redshift Serverless base RPU low; let it auto-pause when idle.
- Set a Redshift Serverless monthly RPU-hour usage limit and alert thresholds.
- Use S3 lifecycle rules and Intelligent-Tiering for Iceberg tables.
- Keep Flink checkpoint retention finite in lower environments.
- Stop local Docker and non-prod EMR clusters when not testing.
- Track actual MSK throughput before committing to fixed broker sizing.